# GWASLab Harmonization Workflow

This notebook walks through **harmonizing GWAS summary statistics against
reference resources** using [GWASLab](https://cloufield.github.io/gwaslab/),
on the toy dataset from the official
[harmonization workflow tutorial](https://cloufield.github.io/gwaslab/harmonization_workflow/).

It mirrors the reusable pipeline in `src/gwaslab_harmonization/` so you can see
exactly which GWASLab operations the tool runs.

> **This notebook has not been executed in the repository's development
> environment** (GWASLab's `pysam` dependency would not build there and the
> reference resources are multi-gigabyte). Any output shown or described below
> is illustrative of the *tooling*, not a measured scientific result. Run it
> locally after installing GWASLab and downloading the references. See
> `docs/decisions.md`.

**What harmonization does:** it reconciles a summary-statistics file with the
reference genome and reference allele frequencies — checking REF/ALT alleles,
resolving strand for palindromic SNPs, handling indels, and recording a status
code per variant. It needs reference data to do that; without it, the run is
only standardization / basic checks.

## 0. Setup

Install the workflow and its dependencies (from the repository root):

```bash
pip install -e ".[notebook]"
```

In [ ]:
from pathlib import Path

import gwaslab as gl

from gwaslab_harmonization import (
    format_config_summary,
    get_version_info,
    load_config,
    run_harmonization,
    write_run_manifest,
)

# Resolve the repository root whether the notebook is run from the repo root
# or from the notebooks/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "harmonization.yaml").is_file():
    REPO_ROOT = REPO_ROOT.parent

print("Repository root:", REPO_ROOT)
print("Versions:", get_version_info())

## 1. Load and validate the configuration

All settings live in `config/harmonization.yaml`. Loading validates the file
and resolves relative paths against the repository root *before* any GWASLab
call runs — mistakes fail loudly and early.

In [ ]:
config = load_config(REPO_ROOT / "config" / "harmonization.yaml", base_dir=REPO_ROOT)
print(format_config_summary(config))

Notice the warning when no reference FASTA / inference VCF is configured. That
combination means the run is **not** full harmonization — it is basic checks
only. We will fix that in step 3.

## 2. Inspect the input

The toy dataset has 15 variants. Its `NOTE` column describes the issue each one
was designed to exercise (clean match, allele flip, palindromic SNP, indel,
ambiguous cases, …). We carry it through harmonization via `input.other_columns`.

In [ ]:
import pandas as pd

toy = pd.read_csv(config.input.path, sep="\t")
print("Shape:", toy.shape)
toy[["SNPID", "EA", "NEA", "EAF", "NOTE"]]

## 3. Reference resources (required for true harmonization)

Reference data is large and is **not** bundled with this repository. GWASLab
does not download it implicitly: `gl.get_path(name)` only looks up a local path
(returning `False` if missing), whereas `gl.download_ref(name)` downloads it.

The cell below downloads the hg19 FASTA and the EAS 1000 Genomes allele
frequencies. **This is slow and uses several gigabytes** — run it once. If you
already have the files, skip it and set the paths in the config instead.

In [ ]:
# Run once. Downloads to GWASLab's cache (~/.gwaslab by default).
fasta_path = gl.download_ref("ucsc_genome_hg19")
vcf_path = gl.download_ref("1kg_eas_hg19")

print("FASTA        :", fasta_path)
print("Inference VCF:", vcf_path)

### Point the configuration at the downloaded resources

The config is frozen (immutable) by design, so we build a fresh config mapping
with the reference paths filled in. This is what you would instead set directly
in `config/harmonization.yaml`.

In [ ]:
import yaml

from gwaslab_harmonization import validate_config

raw = yaml.safe_load((REPO_ROOT / "config" / "harmonization.yaml").read_text())
raw["reference"]["fasta_path"] = str(fasta_path)
raw["reference"]["inference_vcf_path"] = str(vcf_path)
raw["reference"]["allele_frequency_field"] = "AF"

config = validate_config(raw, source_path="<notebook>", base_dir=REPO_ROOT)
print(format_config_summary(config))

## 4. Run harmonization

`run_harmonization` performs exactly two GWASLab calls under the hood:

1. `gl.Sumstats(path, fmt=..., build=..., other=[...])` — read + normalize.
2. `Sumstats.harmonize(basic_check=..., ref_seq=..., ref_infer=...,
   ref_alt_freq=..., threads=..., ...)` — the harmonization.

Runs with `save=False` here so the notebook does not write output by default;
set `save=True` to write to `results/`.

In [ ]:
result = run_harmonization(config, save=False)

print("Variants in :", result.n_variants_in)
print("Variants out:", result.n_variants_out)
print("Delta       :", result.variant_delta)
print("Build       :", result.genome_build, f"(code {result.gwaslab_build_code})")

## 5. Inspect the results

GWASLab writes a 7-digit `STATUS` code per variant (`1` = check passed). The
seven checks are: build, rsID, CHR&POS, EA/NEA, REF-NEA alignment,
palindromic/strand, and indel. See `docs/results.md` for the full table.

We re-create the `Sumstats` object here so you can call GWASLab's own reporting
helpers (`summary`, `lookup_status`).

In [ ]:
from gwaslab_harmonization import build_harmonize_kwargs, build_sumstats_kwargs

mysumstats = gl.Sumstats(str(config.input.path), **build_sumstats_kwargs(config))
mysumstats.harmonize(**build_harmonize_kwargs(config))

mysumstats.summary()
mysumstats.lookup_status(status="STATUS")

### Save the harmonized table

Uncomment to write the output to `results/` (git-ignored).

In [ ]:
# config.output.directory.mkdir(parents=True, exist_ok=True)
# mysumstats.to_format(str(config.output.path), build=config.gwaslab_build_code)
# print("Wrote:", config.output.path)

## 6. Record a reproducibility manifest

The manifest captures the input, the reference resources actually used, the
exact GWASLab arguments, and software versions — so a result can be traced back
to a specific run.

In [ ]:
manifest_path = write_run_manifest(
    result, config, manifest_path=REPO_ROOT / "results" / "notebook_run_manifest.json"
)
print(manifest_path.read_text()[:800])

## 7. Summary and next steps

- Harmonization maps cleanly onto GWASLab: read with `Sumstats`, harmonize with
  `harmonize`, write with `to_format`.
- Reference resources are what make it *harmonization* rather than
  standardization. Without them, only basic checks run — and the workflow warns
  you.
- API caveats worth remembering: use `threads=` (not the removed `n_cores=`),
  and `download_ref()` (not `get_path()`) to fetch resources.

See `docs/` for deeper detail: `workflow.md`, `reference_data.md`,
`results.md`, and `decisions.md`.